# Calculation Reynolds number steady state and timescales

In [ ]:
#update reading in packages when rerunning this cell
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt

# plotstyle: 
plt.style.use('../python_style_Meike.mplstyle')
import sys
sys.path.append("/nethome/4291387/Maxey_Riley_advection/Maxey_Riley_advection/src")
import particle_characteristics_functions as pcf

In [ ]:
def Rep_geostrophic_uslip_stst_dim(B, d, f, nu, U,Rep):
    """
    calculates particle reynolds number based on geostrophic balance of
    steady state flow given by U=(U,0). 
    - B is buoyancy
    - d is particle diameter (assuming sphere)
    - f is coriolis paramter 
    - nu is kinematic viscosity
    - U is magnitude flow velocity
    """
    tstokes = pcf.stokes_relaxation_time(d,nu,B)
    crep = pcf.factor_drag_white1991(Rep)
    denominator = (1+2*B)*(crep**2 + tstokes**2 * f**2 )
    uslip = 2* (1-B) * tstokes**2 *  f**2 * U / denominator 
    vslip = 2 * (1-B) * crep * tstokes *  f * U / denominator 
    Uslip = np.sqrt(uslip**2+vslip**2)
    return Uslip*d/nu


# def Rep_geostrophic_uslip_stst_dim(B, d, f, nu, U,Rep):
#     a = d/2
#     R = 2 / (1+ 2 *B)
#     crep = pcf.factor_drag_white1991(Rep)
#     epsilon =   2 / 9 * (d/2 )**2 / (nu * crep)  * 1/R
#     norm =((1/epsilon)**2 + f**2) 
#     uslip = 2* (1-B) * f**2 * U / (   (1 + 2* B) *  norm ) 
#     vslip = 18 * (1-B) * 1 * nu *  f * U / (  (1+2*B)**2 * a**2 *  norm ) 
#     Uslip = np.sqrt(uslip**2+vslip**2)
#     return Uslip*d/nu

def tau_drag(tau,Rep):
    """
    calculates drag time, i.e. timescale
    associated with drag force. Expression reduces
    to stokes relaxation time for Rep=0
    """
    c_rep = pcf.factor_drag_white1991(Rep)
    return tau/c_rep



In [ ]:
# properties drifters
d = 0.25 # m diameter outer ring
d_in = 0.2 # m (estimated) dimater inner ring, needed to calculate volume
h = 0.041  # m (heigth drifter)
m = 0.905 # kg (mass drifter)
omega_earth =  7.2921e-5 #[rad/sec]
f = 2 * omega_earth * np.sin(54/180 * np.pi) # coriolis at 54 deg N 

#properties water
av_temp_NWES = 11.983276
av_salinity_NWES = 34.70449
rho_water = 1027 # kg/m3 
dynamic_viscosity_water = pcf.dynamic_viscosity_Sharqawy(av_temp_NWES,av_salinity_NWES/1000)
kinematic_viscosity_water = dynamic_viscosity_water / rho_water

# simulation values drifters
B = pcf.buoyancy_drifter(diameter  = d_in, heigth = h, mass = m, density_fluid = rho_water)
tau = pcf.stokes_relaxation_time(d, kinematic_viscosity_water, B)

For simulations with the MRG equations we need to set the stokes time and particle reynolds number (or keep is flexible)
Below I show how you can calculate the stokes relaxation time (tau) for a range of diamters and how you can estimate the particle reynolds number for a range of diameters (note that this is an estimate for which I asume a typical flow velocity of 1 m/s)

In [ ]:
# stokes time tau and estimate for particle reynolds number as function of diamter
dlist = np.arange(0.001,2,0.001) # range of diameters in meters
Uflow = 0.25337228# m/s (meain flow velocity NWES) 1 # m/s (typical tidal flow fvelocity)
TM2=12.4206*3600
Lflow = Uflow*TM2/4
taulist = pcf.stokes_relaxation_time(dlist, kinematic_viscosity_water, B)
Rep_geostrophic_estimate=[]
for d in dlist:
    def fie(Rep):
        return Rep_geostrophic_uslip_stst_dim(B,d,f,kinematic_viscosity_water,Uflow,Rep)-Rep
    Rep_geostrophic_estimate.append(fsolve(fie,300)[0])
Rep_geostrophic_estimate = np.asarray(Rep_geostrophic_estimate)

In [ ]:
fig, ax = plt.subplots()
ax.plot(dlist,taulist,color='black')
ax2=ax.twinx()
ax2.plot(dlist,Rep_geostrophic_estimate,color='firebrick')
ax.set_xlabel('diameter [m]')
ax.set_ylabel('$\\tau_{\\mathrm{stokes}}$ [s]')
ax2.set_ylabel('Re$_p$',color='firebrick')

ax2.spines['right'].set_color('firebrick')
ax2.yaxis.label.set_color('firebrick')
ax2.tick_params(axis='y', colors='firebrick')

print(f'Rep ( d = {dlist[249]} m) = {Rep_geostrophic_estimate[249]}: c(Rep) = {pcf.factor_drag_white1991(Rep_geostrophic_estimate[249])}')



In [ ]:
# epsilon value:
epsilon = taulist[249]*Uflow/Lflow
print(epsilon)

In [ ]:
# timesscales associated with history kernel
diameterlist=[0.0025,0.025,0.25]
c2Kim = 0.126
for d in diameterlist:
    #find Rep value
    def fie(Rep):
        return Rep_geostrophic_uslip_stst_dim(B,d,f,kinematic_viscosity_water,Uflow,Rep)-Rep
    Repsol = fsolve(fie,300)[0]
    Uslip = Repsol * kinematic_viscosity_water / d
    tdif = pcf.diffusion_time(d,kinematic_viscosity_water)
    tOseen = pcf.Oseen_time(Uslip, kinematic_viscosity_water)
    tMeiAdrian = pcf.MeiAdrian_time(c2Kim,tOseen,tdif)
    
    print(f'd = {d:.03f}  Rep = {Repsol:.03f} tdiff = {tdif:.03f} tOseen = {tOseen:.03f} tMeiAdrian = {tMeiAdrian:03f}')



In [ ]:
fig, ax = plt.subplots() 
ax.plot(dlist,tau_drag(taulist,Rep_geostrophic_estimate))
ax.set_xlabel('diameter [m]')
ax.set_ylabel('$\\tau_{\\mathrm{drag}}$ [s]')